# Evaluate RPD with mDeBERTa-v3-base

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
MODEL_NAME = "mdeberta-v3-base"
NB_ID = "21.55"

In [ ]:
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

## Data Hash

```
./data/RP_train.csv           : 97318e
./data/RP_valid.csv           : 78336b
./data/glossdict.json         : 5ef588
```

In [ ]:
rptest = "./dotted-wsd/GlossBERT/data/RP_valid.csv"
rptrain = "./dotted-wsd/GlossBERT/data/RP_train.csv"
gdpath = "./dotted-wsd/GlossBERT/data/glossdict.json"

CHECK_DATA_HASH = True
if CHECK_DATA_HASH:
    import hashlib
    from pathlib import Path

    for data_path in (rptrain, rptest, gdpath):
        hasher = hashlib.sha1()
        hasher.update(Path(data_path).read_bytes())
        h = hasher.digest().hex()[:6]
        print(f"{data_path:<30s}: {h}")

In [ ]:
rp_valid = pd.read_csv(rptest)
rp_valid.shape

In [ ]:
rp_valid.head()

## Evaluation

The evaluation accuracy computed with 20.55 is:
```
rp_eval = evaluate(model, rp_evalloader)
```

In [ ]:
PROJECT_ROOT = "./dotted_wsd_for_mdebertabase"
MODEL_NAME = "mDeBERTa-v3-base"

In [ ]:
from dotted_wsd_for_mdebertabase import DottedWsdTagger

DeBERTatagger = DottedWsdTagger()

In [ ]:
DeBERTatagger.model

In [ ]:
rp_mask = rp_valid.apply(lambda r: r["RP Class"] in r["dot_obj"], axis=1)
rp_valid = rp_valid.loc[rp_mask]
input_text = rp_valid.iloc[0].Sentence
dot_obj = rp_valid.iloc[0].dot_obj
input_text, dot_obj

In [ ]:
rp_valid.shape

In [ ]:
ref_labels = rp_valid["RP Class"]
rp_labels = sorted(ref_labels.unique().tolist())

In [ ]:
rp_valid.head()

## With dotted-type hints 

In [ ]:
from tqdm.auto import tqdm

predictions = []
for _, row in tqdm(rp_valid.iterrows()):
    input_text = row.Sentence
    dot_obj = row.dot_obj
    ex_pred, _ = DeBERTatagger.dotted_tag(input_text, dot_obj)
    predictions.append((ex_pred.pred_class, ex_pred.prob))

In [ ]:
pred_dotted = [x[0] for x in predictions]
prob_dotted = [x[1] for x in predictions]

In [ ]:
# mDeBERTa-v3-base results
print(classification_report(ref_labels, pred_dotted))

In [ ]:
# GlossBERT results
# print(classification_report(ref_labels, pred_dotted))

In [ ]:
# mDeBERTa-v3-base results
acc_dotted = accuracy_score(ref_labels, pred_dotted)
f1w_dotted = f1_score(ref_labels, pred_dotted, average="weighted")
conf_mat = confusion_matrix(ref_labels, pred_dotted)
ConfusionMatrixDisplay(conf_mat, display_labels=rp_labels).plot(xticks_rotation=45)

In [ ]:
import joblib

rppred_path = f"./data/{MODEL_NAME}_RPvalid_alltypes_with_hints_0.51.pkl"
joblib.dump(predictions, rppred_path)

## Without dotted-type hints

In [ ]:
from tqdm.auto import tqdm

predictions = []
for _, row in tqdm(rp_valid.iterrows()):
    input_text = row.Sentence
    dot_obj = row.dot_obj
    ex_pred, _ = DeBERTatagger.rp_tag(input_text)
    predictions.append((ex_pred.pred_class, ex_pred.prob))

In [ ]:
pred_alltypes = [x[0] for x in predictions]
prob_alltypes = [x[1] for x in predictions]

In [ ]:
# mDeBERTa-v3-base results (without dotted-type hints)
acc_alltypes = accuracy_score(ref_labels, pred_alltypes)
f1w_alltypes = f1_score(ref_labels, pred_alltypes, average="weighted")
print(classification_report(ref_labels, pred_alltypes))

In [ ]:
import joblib

rppred_path = f"./data/{MODEL_NAME}_RPvalid_alltypes_without_dotted_hints_0.37.pkl"
joblib.dump(predictions, rppred_path)

In [ ]:
# mDeBERTa-v3-base results (without dotted-type hints)
conf_mat_nohints = confusion_matrix(ref_labels, pred_alltypes)
ConfusionMatrixDisplay(conf_mat_nohints, display_labels=rp_labels).plot(xticks_rotation=45)

## A dummy classifier

In [ ]:
rptrain = "./data/RP_train.csv"
rptrainset = pd.read_csv(rptrain)
most_freq_clf = (
    rptrainset.groupby("dot_obj")
    .apply(lambda x: x.value_counts("RP Class").index.values[0])
    .to_dict()
)
rptrainset.head()

In [ ]:
pred_dummy = [most_freq_clf[x] for x in rp_valid["dot_obj"]]
acc_dummy = accuracy_score(ref_labels, pred_dummy)
f1w_dummy = f1_score(ref_labels, pred_dummy, average="weighted")
print(classification_report(ref_labels, pred_dummy))

## Outputs

In [ ]:
rp_valid_preds = rp_valid.assign(
    pred_dotted=pred_dotted,
    prob_dotted=prob_dotted,
    pred_alltypes=pred_alltypes,
    prob_alltypes=pred_alltypes,
    pred_dummy=pred_dummy,
)

In [ ]:
pred_path = "./data/rp_valid_preds.csv"
rp_valid_preds.to_csv(pred_path)

In [ ]:
import json
import os

os.makedirs("../data/metrics", exist_ok=True)
metric_path = "../data/metrics/rp-eval-preds.json"
with open(metric_path, "w") as fout:
    json.dump(
        dict(
            acc_dotted=acc_dotted,
            f1w_dotted=f1w_dotted,
            acc_alltypes=acc_alltypes,
            f1w_alltypes=f1w_alltypes,
            acc_dummy=acc_dummy,
            f1w_dummy=f1w_dummy,
        ),
        fout,
    )

## Out hash
```
../data/rp_valid_preds.csv 4917ed
../data/metrics/rp-eval-preds.json 8476e9
```

In [ ]:
for path_x in (pred_path, metric_path):
    h = hashlib.sha1()
    h.update(Path(path_x).read_bytes())
    print(path_x, h.hexdigest()[:6])